In [ ]:
import glob
import pandas as pd
import numpy as np
import plotly.express as px

def df_to_clipboard(df, index=False):
    print(df.to_markdown(index=index))

# Load files

In [ ]:
# JSONL from LLM pipeline (1.0-schema_discovery.ipynb)
files = glob.glob("data/schema_discovery_results_*.jsonl")
files

In [ ]:
data = pd.concat([pd.read_json(x, lines=True) for x in files]).reset_index(drop=True)
data

In [ ]:
data[data["error"].notna()]

In [ ]:
data["snapshot_type"] = data["snapshot_id"].apply(lambda x: x.split("_")[-2])
data.groupby(["source", "snapshot_type"]).agg(
    count=("source", "count"),
)

# Preprocessing

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df = data[["snapshot_id", "snapshot_type", "source", "model", "parsed_output"]].copy()
df["parsed_output"] = df["parsed_output"].apply(
    lambda x: x.get("fields", []) if isinstance(x, dict) else []
)

df = df.explode("parsed_output").reset_index(drop=True)
df = df.join(pd.json_normalize(df["parsed_output"])).drop(columns="parsed_output")

df.head()

In [ ]:
# df.to_csv("outputs/2.0-discovery_results.csv", index=False)

# Source citation

In [ ]:
source_df = df[df["metadata_field"].str.contains("source")].copy()
excl_list = [
    x
    for x in source_df["metadata_field"].unique()
    if any(sub in x for sub in ["financing", "funding"])
]
source_df = source_df[~source_df["metadata_field"].isin(excl_list)]
source_df

In [ ]:
source_df["metadata_field"].value_counts()

In [ ]:
df_to_clipboard(source_df["metadata_field"].value_counts(), index=True)

In [ ]:
source_df["observed_value"].value_counts()

In [ ]:
source_df[source_df["observed_value"] == "Not identifiable from this snapshot"]

# Top metadata fields

In [ ]:
df["metadata_field"].value_counts().head(10)

# Geographic scope

In [ ]:
geo = df[df["metadata_field"] == "geographic_scope"].copy()

In [ ]:
geo["snapshot_id"].nunique()

In [ ]:
geo["source_level"].value_counts()

# Population group

In [ ]:
pop = df[df["metadata_field"] == "population_group"].copy()

In [ ]:
pop["snapshot_id"].nunique()